# Connect-4 PPO Evaluation Suite

This notebook now focuses on two headline numbers:

1. **Global score**: a depth-weighted average of benchmark win-rates.
2. **Elo-like league rating**: a pool rating computed from head-to-head results vs baseline agents.

Important notes:

- Global score is **not** a probability, and it is **not** Elo. It is a weighted benchmark summary.
- Deep lookahead opponents receive much larger weights, so performance against stronger opponents matters more.
- For deterministic agents, replaying the same empty-board game again and again adds very little information. To reduce opening bias, benchmark and league matches sweep a small set of opening prefixes instead of testing only the empty starting position.
- The league rating is intentionally still a fake / practical Elo-like number, but it is fitted from aggregate match scores in a more stable way than a single chronological Elo pass.


In [1]:
MODEL_PATH = "PPO_Models/PPO_852X.pt" 
#MODEL_PATH = "PPO_Models/X_1.pt" 
#MODEL_PATH = "PPO_Models/SOUP_9.pt" 
#MODEL_PATH = "SupervisedModels/BASE_9.pt" 

In [2]:
# --- Imports & device ---
import os
import re
import math
import time
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Iterable, Any
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path
import torch
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 666
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("DEVICE:", DEVICE, "| seed:", SEED)
start_time = time.time()

DEVICE: cuda | seed: 666


In [3]:
from C4.eval_oppo_dict import EVALUATION_OPPONENTS, EVAL_CFG
from C4.fast_connect4_lookahead import Connect4Lookahead
from C4.CNet192 import CNet192, load_cnet192
from PPO.actor_critic import ActorCritic, TransferCfg

LA_SHARED = Connect4Lookahead()
LA_SHARED.OPENING_RANDOM = False

BENCHMARK_OPPONENTS = dict(EVALUATION_OPPONENTS)
for k in ["Lookahead-7", "Lookahead-9", "Lookahead-11", "Lookahead-13"]:
    if k in BENCHMARK_OPPONENTS:
        BENCHMARK_OPPONENTS[k] = max(int(BENCHMARK_OPPONENTS[k]), 20)

# Evaluation opening suite.
# Rationale:
# - repeated empty-board games are almost useless for deterministic agents,
# - a tiny opening sweep gives the benchmark and league much better coverage,
# - paired seat swaps are still used on top.
OPENING_PREFIXES = [
    (),
    (3,),
    (2,),
    (4,),
    (3, 2),
    (3, 4),
    (2, 3),
    (4, 3),
    (3, 2, 3),
    (3, 4, 3),
]

MIN_GAMES_PER_SERIES = 2 * len(OPENING_PREFIXES)
EVAL_TEMPERATURE = 0.05   # nearly greedy, keeps PPO eval much less noisy

print("OK: imported Connect4Lookahead, CNet192/load_cnet192, ActorCritic")
print("Opening prefixes:", len(OPENING_PREFIXES), "| min games per paired series:", MIN_GAMES_PER_SERIES)


OK: imported Connect4Lookahead, CNet192/load_cnet192, ActorCritic
Opening prefixes: 10 | min games per paired series: 20


In [4]:

ROWS, COLS = 6, 7

def empty_board() -> np.ndarray:
    return np.zeros((ROWS, COLS), dtype=np.int8)

def legal_actions(board: np.ndarray) -> List[int]:
    return [c for c in range(COLS) if board[0, c] == 0]

def apply_action(board: np.ndarray, col: int, mark: int) -> np.ndarray:
    if board[0, col] != 0:
        raise ValueError(f"Column {col} is full")
    out = board.copy()
    for r in range(ROWS - 1, -1, -1):
        if out[r, col] == 0:
            out[r, col] = np.int8(mark)
            return out
    raise RuntimeError("apply_action fell through")

def is_full(board: np.ndarray) -> bool:
    return bool(np.all(board[0, :] != 0))

def other_mark(mark: int) -> int:
    return 2 if mark == 1 else 1

def has_winner(board: np.ndarray, mark: int) -> bool:
    return bool(LA_SHARED.has_four(board, mark))

def terminal_outcome(board: np.ndarray) -> Optional[int]:
    if has_winner(board, 1): return 1
    if has_winner(board, 2): return 2
    if is_full(board): return 0
    return None


def board_to_pov_scalar(board012: np.ndarray, mark: int) -> np.ndarray:
    """(6,7) int8 in {-1,0,+1}, current player=+1."""
    b = np.asarray(board012, dtype=np.int8)
    me = np.int8(mark)
    opp = np.int8(other_mark(mark))
    pov = np.zeros_like(b, dtype=np.int8)
    pov[b == me] = 1
    pov[b == opp] = -1
    return pov


In [5]:
# --- Agent interface + baselines ---

class Agent:
    name: str = "Agent"
    def begin_episode(self, seed: int = 0) -> None:
        pass
    def select_action(self, board: np.ndarray, mark: int, rng: np.random.Generator) -> int:
        raise NotImplementedError

class RandomAgent(Agent):
    def __init__(self, name="Random"):
        self.name = name
    def select_action(self, board, mark, rng):
        legal = legal_actions(board)
        return int(rng.choice(legal)) if legal else 0

class LeftmostAgent(Agent):
    def __init__(self, name="Leftmost"):
        self.name = name
    def select_action(self, board, mark, rng):
        legal = legal_actions(board)
        return int(min(legal)) if legal else 0

class CenterAgent(Agent):
    def __init__(self, name="Center"):
        self.name = name
    def select_action(self, board, mark, rng):
        for c in [3,4,2,5,1,6,0]:
            if board[0, c] == 0:
                return int(c)
        return 0

class LookaheadAgent(Agent):
    def __init__(self, depth: int, name: Optional[str]=None, cfg: Optional[dict]=None):
        self.depth = int(depth)
        self.name = name or f"Lookahead-{depth}"
        self.la = Connect4Lookahead()
        self.la.OPENING_RANDOM = False
        if cfg:
            for k, v in cfg.items():
                setattr(self.la, k, v)
    def select_action(self, board, mark, rng):
        return int(self.la.n_step_lookahead(board, mark, depth=self.depth))

class ActorCriticAgent(Agent):
    """Wrap your PPO ActorCritic for this evaluation harness."""
    def __init__(self, ac: ActorCritic, name: str = "PPO-AC", temperature: float = 1.0, disable_rule_mixes: bool = True):
        self.ac = ac.to(DEVICE).eval()
        self.name = name
        self.temperature = float(temperature)
        self.disable_rule_mixes = bool(disable_rule_mixes)
        self._ply = 0

    def begin_episode(self, seed: int = 0) -> None:
        self._ply = 0
        torch.manual_seed(int(seed))
        if hasattr(self.ac, "begin_episode"):
            self.ac.begin_episode()
        if self.disable_rule_mixes and hasattr(self.ac, "set_phase_heuristics"):
            # Pure policy evaluation
            self.ac.set_phase_heuristics(center_start=0.0, guard_prob=0.0, win_now_prob=0.0)

    def select_action(self, board: np.ndarray, mark: int, rng: np.random.Generator) -> int:
        legal = legal_actions(board)
        if not legal:
            return 0
        pov = board_to_pov_scalar(board, mark)
        a, logp, v, info = self.ac.act(
            state_np=pov,
            legal_actions=legal,
            temperature=self.temperature,
            ply_idx=self._ply,
        )
        self._ply += 1
        return int(a)


In [6]:
# --- Load PPO checkpoint (save_cnet192 format) ---
def load_actor_critic_from_save_cnet192(path: str) -> ActorCritic:
    ac = ActorCritic.from_cnet192_checkpoint(
        path=path,
        device=DEVICE,
        override_cfg=None,
        transfer=TransferCfg(freeze_conv=False, strict_load=True),
    )
    return ac.to(DEVICE).eval()


ac = load_actor_critic_from_save_cnet192(MODEL_PATH)

# Use a nearly-greedy temperature for evaluation.
# Pure sampling (temperature=1.0) is useful for training, but it adds noise to
# benchmark / league numbers
AGENT_UNDER_TEST: Agent = ActorCriticAgent(
    ac,
    name=os.path.basename(MODEL_PATH),
    temperature=EVAL_TEMPERATURE,
    disable_rule_mixes=True,
)
print("Loaded PPO model:", MODEL_PATH, "| eval temperature:", EVAL_TEMPERATURE)


Loaded PPO model: PPO_Models/PPO_852X.pt | eval temperature: 0.05


In [7]:
# --- Match runner (paired seats + opening sweep) ---

@dataclass
class MatchStats:
    wins: int = 0
    losses: int = 0
    draws: int = 0

    @property
    def games(self) -> int:
        return self.wins + self.losses + self.draws

    @property
    def win_rate(self) -> float:
        return self.wins / self.games if self.games else 0.0

    @property
    def score(self) -> float:
        return (self.wins + 0.5 * self.draws) / self.games if self.games else 0.0


def apply_opening_prefix(opening_moves: Iterable[int]) -> Tuple[np.ndarray, int]:
    board = empty_board()
    mark = 1

    for ply, col in enumerate(opening_moves, start=1):
        legal = legal_actions(board)
        if int(col) not in legal:
            raise ValueError(f"Illegal opening move at ply {ply}: col={col}, opening={tuple(opening_moves)}")

        board = apply_action(board, int(col), mark)

        # Opening prefixes are meant to land on a live position.
        outcome = terminal_outcome(board)
        if outcome is not None:
            raise ValueError(f"Opening prefix already ends the game at ply {ply}: {tuple(opening_moves)}")

        mark = other_mark(mark)

    return board, mark


def play_game(
    agent1: Agent,
    agent2: Agent,
    seed: int,
    opening_moves: Iterable[int] = (),
    max_plies: int = 42,
) -> int:
    rng = np.random.default_rng(seed)
    torch.manual_seed(int(seed))
    agent1.begin_episode(seed=seed)
    agent2.begin_episode(seed=seed + 1)

    board, mark = apply_opening_prefix(tuple(opening_moves))

    for _ply in range(int(np.count_nonzero(board)) + 1, max_plies + 1):
        if mark == 1:
            col = agent1.select_action(board, 1, rng)
            board = apply_action(board, col, 1)
            if has_winner(board, 1):
                return 1
        else:
            col = agent2.select_action(board, 2, rng)
            board = apply_action(board, col, 2)
            if has_winner(board, 2):
                return 2

        if is_full(board):
            return 0
        mark = other_mark(mark)

    return 0


def evaluate_matchup(
    agentA: Agent,
    agentB: Agent,
    n_games: int,
    seed: int = 0,
    alternate_starts: bool = True,
    opening_prefixes: Optional[Iterable[Tuple[int, ...]]] = None,
) -> MatchStats:
    prefixes = [tuple(p) for p in (opening_prefixes if opening_prefixes is not None else [()])]
    if not prefixes:
        prefixes = [()]

    # For paired-seat evaluation we want at least one full opening sweep.
    if alternate_starts:
        n_games = max(int(n_games), 2 * len(prefixes))
    else:
        n_games = max(int(n_games), len(prefixes))

    st = MatchStats()
    games_done = 0
    round_idx = 0

    while games_done < n_games:
        for pref_idx, prefix in enumerate(prefixes):
            if games_done >= n_games:
                break

            pair_seed = seed + 1000 * round_idx + pref_idx

            winner = play_game(agentA, agentB, seed=pair_seed, opening_moves=prefix)
            if winner == 1:
                st.wins += 1
            elif winner == 2:
                st.losses += 1
            else:
                st.draws += 1
            games_done += 1

            if alternate_starts and games_done < n_games:
                winner = play_game(agentB, agentA, seed=pair_seed, opening_moves=prefix)
                if winner == 1:
                    st.losses += 1
                elif winner == 2:
                    st.wins += 1
                else:
                    st.draws += 1
                games_done += 1

        round_idx += 1

    return st


st = evaluate_matchup(
    AGENT_UNDER_TEST,
    RandomAgent(),
    n_games=20,
    seed=SEED,
    alternate_starts=True,
    opening_prefixes=OPENING_PREFIXES,
)
print("Sanity vs Random:", st, "wr=", round(st.win_rate, 3), "score=", round(st.score, 3))


Sanity vs Random: MatchStats(wins=19, losses=1, draws=0) wr= 0.95 score= 0.95


In [8]:
%%time
def opponent_weight(label: str, base: float = 1.4, random_weight: float = 1.0, default_weight: float = 1.0) -> float:
    """
    Global-score weights.

    Convention used here:
    - Random gets a fixed weight of 1.0,
    - Lookahead-k gets weight base**k,
    - everything else falls back to default_weight.

    So with base=1.4:
    - LA-1  counts as 1.4x Random,
    - LA-7  counts as 1.4**7,
    - LA-13 counts as 1.4**13.

    Interpretation:
    this is a *benchmark summary score*, not a true rating system.
    It heavily rewards success against deeper lookahead opponents.

    Practical warning:
    because the weights rise fast, deep opponents need enough games and enough
    opening diversity, otherwise a tiny sample can swing the final score too much.
    """
    s = str(label)
    if "Random" in s:
        return float(random_weight)
    m = re.search(r"(\d+)", s)
    if m:
        return float(base) ** int(m.group(1))
    return float(default_weight)


def global_score_from_results(results: dict, base: float = 1.4) -> float:
    """
    Weighted average of benchmark win-rates.

    Formula:
        G = sum_i weight_i * win_rate_i / sum_i weight_i

    where weight_i is produced by opponent_weight(label, base).

    Nice property:
    - easy to compute,
    - easy to compare across runs,
    - strongly prioritizes harder opponents.

    Not-so-nice property:
    - not calibrated like Elo,
    - very sensitive to the chosen benchmark suite and weights,
    - should be read as "how good was this run on *our* benchmark" and nothing more.
    """
    num = 0.0
    den = 0.0
    for label, stats in results.items():
        wr = float(stats.get("win_rate", 0.0))
        w = opponent_weight(label, base=base)
        num += w * wr
        den += w
    return num / den if den > 1e-12 else float("nan")


def global_score_breakdown(results: dict, base: float = 1.4) -> pd.DataFrame:
    rows = []
    for label, stats in results.items():
        wr = float(stats.get("win_rate", 0.0))
        games = int(stats.get("games", 0))
        w = opponent_weight(label, base=base)
        rows.append({
            "opponent": label,
            "games": games,
            "win_rate": wr,
            "weight": w,
            "weighted_win_rate": w * wr,
        })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    total_weight = float(df["weight"].sum())
    df["weight_share"] = df["weight"] / total_weight
    df["global_contribution"] = df["weighted_win_rate"] / total_weight
    return df.sort_values("weight", ascending=False).reset_index(drop=True)


def build_opponent(label: str) -> Agent:
    if label == "Random":
        return RandomAgent()
    if label == "Leftmost":
        return LeftmostAgent()
    if label == "Center":
        return CenterAgent()
    if label.startswith("Lookahead-"):
        d = int(label.split("-")[-1])
        return LookaheadAgent(depth=d, name=label)
    raise ValueError(label)


def run_current_benchmark(agent: Agent, seed: int = 0) -> dict:
    results = {}
    items = list(BENCHMARK_OPPONENTS.items())

    for label, n in tqdm(items, desc="Benchmark opponents"):
        opp = build_opponent(label)
        n_games = max(int(n), MIN_GAMES_PER_SERIES)
        st = evaluate_matchup(
            agent,
            opp,
            n_games=n_games,
            seed=seed,
            alternate_starts=True,
            opening_prefixes=OPENING_PREFIXES,
        )

        results[label] = {
            "wins": st.wins,
            "losses": st.losses,
            "draws": st.draws,
            "games": st.games,
            "win_rate": st.win_rate,
            "score": st.score,
        }

        tqdm.write(
            f"{agent.name:>18} vs {label:<12} | wr={st.win_rate:.3f} score={st.score:.3f} (n={st.games})"
        )

    return results


results = run_current_benchmark(AGENT_UNDER_TEST, seed=SEED)
G = global_score_from_results(results, base=1.4)
df_global = global_score_breakdown(results, base=1.4)
print("\nGlobal score:", round(G, 3))


Benchmark opponents:   0%|          | 0/13 [00:00<?, ?it/s]

       PPO_852X.pt vs Random       | wr=0.985 score=0.985 (n=200)
       PPO_852X.pt vs Lookahead-1  | wr=0.750 score=0.750 (n=100)
       PPO_852X.pt vs Lookahead-2  | wr=0.750 score=0.750 (n=100)
       PPO_852X.pt vs Lookahead-3  | wr=0.600 score=0.600 (n=100)
       PPO_852X.pt vs Lookahead-4  | wr=0.500 score=0.500 (n=100)
       PPO_852X.pt vs Lookahead-5  | wr=0.560 score=0.560 (n=50)
       PPO_852X.pt vs Lookahead-6  | wr=0.333 score=0.333 (n=24)
       PPO_852X.pt vs Lookahead-7  | wr=0.600 score=0.625 (n=20)
       PPO_852X.pt vs Lookahead-9  | wr=0.500 score=0.525 (n=20)
       PPO_852X.pt vs Lookahead-11 | wr=0.450 score=0.450 (n=20)
       PPO_852X.pt vs Lookahead-13 | wr=0.400 score=0.400 (n=20)
       PPO_852X.pt vs Leftmost     | wr=0.900 score=0.900 (n=100)
       PPO_852X.pt vs Center       | wr=1.000 score=1.000 (n=200)

Global score: 0.459
CPU times: total: 6min 55s
Wall time: 7min 4s


## Benchmark notes

The regret / blunder section was removed on purpose.

For this notebook the two headline outputs are now:

- **Global score**, for a compact benchmark summary.
- **Elo-like league rating**, for a relative pool score.

That keeps the notebook much easier to read, and avoids giving too much importance to metrics that were not proving very actionable in practice.


In [9]:
%%time
# --- Cached Elo-like league (opening sweep + batch fit) ---

# 1) it does NOT rely on repeated empty-board games only,
# 2) it enforces a sensible minimum series size via the opening sweep,
# 3) it fits an order-stable Elo-like rating from aggregate scores,
#    instead of doing one single chronological Elo pass.
#
# This is still not "real Elo", but it is much less twitchy.

LEAGUE_CACHE_VERSION = "v2_openings_batchfit"
CACHE_DIR = Path("league_cache") / LEAGUE_CACHE_VERSION
CACHE_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_CACHE = CACHE_DIR / "baseline_pairs.pkl"

def _sanitize(s: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9._-]+", "_", str(s))
    return s[:160]

def _la_depth_from_name(name: str) -> int:
    # Supports "Lookahead-7" and "LA-7"
    m = re.search(r"(?:Lookahead-|LA-)(\d+)", str(name))
    return int(m.group(1)) if m else 0

def games_for_pair(nameA: str, nameB: str) -> int:
    da = _la_depth_from_name(nameA)
    db = _la_depth_from_name(nameB)
    dmax = max(da, db)
    dmin = min(da, db)

    # Deep-vs-deep comparisons are where funny orderings are most likely if the
    # sample is too thin, so they get the most coverage.
    if dmin >= 11:
        return max(MIN_GAMES_PER_SERIES, 32)
    if dmin >= 9:
        return max(MIN_GAMES_PER_SERIES, 24)
    if dmax >= 11:
        return max(MIN_GAMES_PER_SERIES, 24)
    if dmax >= 9:
        return max(MIN_GAMES_PER_SERIES, 20)
    if dmax >= 7:
        return max(MIN_GAMES_PER_SERIES, 16)
    return max(MIN_GAMES_PER_SERIES, 12)

def run_round_robin_cached(
    agents: list,
    seed: int,
    desc: str,
) -> pd.DataFrame:
    rows = []
    pairs = [(i, j) for i in range(len(agents)) for j in range(i + 1, len(agents))]
    for (i, j) in tqdm(pairs, desc=desc, total=len(pairs)):
        A, B = agents[i], agents[j]
        n_games = games_for_pair(A.name, B.name)
        st = evaluate_matchup(
            A,
            B,
            n_games=n_games,
            seed=seed + 5000 * (i * 100 + j),
            alternate_starts=True,
            opening_prefixes=OPENING_PREFIXES,
        )
        rows.append((A.name, B.name, st.wins, st.losses, st.draws, n_games))
    return pd.DataFrame(rows, columns=["A", "B", "winsA", "lossesA", "draws", "games"])

def run_vs_baseline_cached(
    agent_under_test: Agent,
    baseline_agents: list,
    seed: int,
    desc: str,
) -> pd.DataFrame:
    vs_path = CACHE_DIR / f"vs__{_sanitize(agent_under_test.name)}.pkl"
    if vs_path.exists():
        df = pd.read_pickle(vs_path)
        print(f"[cache] loaded vs-baseline: {vs_path} ({len(df)} rows)")
        return df

    rows = []
    for j, opp in enumerate(tqdm(baseline_agents, desc=desc, total=len(baseline_agents))):
        n_games = games_for_pair(agent_under_test.name, opp.name)
        st = evaluate_matchup(
            agent_under_test,
            opp,
            n_games=n_games,
            seed=seed + 10000 * j,
            alternate_starts=True,
            opening_prefixes=OPENING_PREFIXES,
        )
        rows.append((agent_under_test.name, opp.name, st.wins, st.losses, st.draws, n_games))

    df = pd.DataFrame(rows, columns=["A", "B", "winsA", "lossesA", "draws", "games"])
    df.to_pickle(vs_path)
    print(f"[cache] saved vs-baseline: {vs_path} ({len(df)} rows)")
    return df

def elo_like_ratings_from_league(
    df_league: pd.DataFrame,
    init: float = 1000.0,
    k: float = 32.0,
    epochs: int = 300,
    tol: float = 1e-6,
) -> pd.DataFrame:
    agents = sorted(set(df_league["A"]).union(df_league["B"]))
    R = {a: float(init) for a in agents}

    def expected(ra, rb):
        return 1.0 / (1.0 + 10.0 ** ((rb - ra) / 400.0))

    for _epoch in tqdm(range(epochs), desc="Batch Elo-like fit"):
        delta = {a: 0.0 for a in agents}

        for row in df_league.itertuples(index=False):
            a, b, winsA, lossesA, draws, games = row
            winsA = int(winsA)
            lossesA = int(lossesA)
            draws = int(draws)
            games = int(games)

            if games <= 0:
                continue

            score_a = (winsA + 0.5 * draws) / games
            exp_a = expected(R[a], R[b])

            # More games => higher confidence, but with a soft cap so giant series
            # do not completely dominate the pool.
            weight = games / (games + 8.0)
            step = k * weight * (score_a - exp_a)

            delta[a] += step
            delta[b] -= step

        max_step = max(abs(v) for v in delta.values()) if delta else 0.0

        for a in agents:
            R[a] += delta[a]

        # Re-center so the pool keeps the requested baseline level.
        mean_r = sum(R.values()) / len(R) if R else init
        for a in agents:
            R[a] = R[a] - mean_r + init

        if max_step < tol:
            break

    return (
        pd.DataFrame([{"agent": a, "elo_like": R[a]} for a in agents])
        .sort_values("elo_like", ascending=False)
        .reset_index(drop=True)
    )

# ---------------- Build baseline league (cached) ----------------
baseline_agents = [
    RandomAgent(),
    LeftmostAgent(),
    CenterAgent(),
] + [LookaheadAgent(d) for d in range(1, 14)]

if BASELINE_CACHE.exists():
    df_base = pd.read_pickle(BASELINE_CACHE)
    print(f"[cache] loaded baseline: {BASELINE_CACHE} ({len(df_base)} rows)")
else:
    df_base = run_round_robin_cached(
        baseline_agents,
        seed=SEED + 30,
        desc="Baseline round-robin (cached)",
    )
    df_base.to_pickle(BASELINE_CACHE)
    print(f"[cache] saved baseline: {BASELINE_CACHE} ({len(df_base)} rows)")

# ---------------- Evaluate new agent only vs baseline (cached per agent) ----------------
df_vs = run_vs_baseline_cached(
    AGENT_UNDER_TEST,
    baseline_agents,
    seed=SEED + 123,
    desc="Under-test vs baseline",
)

# ---------------- Combine + compute Elo-like rating ----------------
df_league = pd.concat([df_base, df_vs], ignore_index=True)
df_rating = elo_like_ratings_from_league(df_league, init=1000.0, k=32.0, epochs=300)

df_rating


[cache] loaded baseline: league_cache\v2_openings_batchfit\baseline_pairs.pkl (120 rows)


Under-test vs baseline:   0%|          | 0/16 [00:00<?, ?it/s]

[cache] saved vs-baseline: league_cache\v2_openings_batchfit\vs__PPO_852X.pt.pkl (16 rows)


Batch Elo-like fit:   0%|          | 0/300 [00:00<?, ?it/s]

CPU times: total: 11min 51s
Wall time: 12min 6s


,agent,elo_like
0,Lookahead-13,1472.467573
1,Lookahead-12,1439.076531
2,Lookahead-11,1404.553875
3,Lookahead-10,1358.967715
4,Lookahead-9,1329.260544
5,Lookahead-7,1248.502209
6,Lookahead-8,1223.545751
7,PPO_852X.pt,1195.016817
8,Lookahead-6,1194.629500
9,Lookahead-5,1186.480792


In [10]:
df_league

,A,B,winsA,lossesA,draws,games
0,Random,Leftmost,6,14,0,20
1,Random,Center,1,19,0,20
2,Random,Lookahead-1,0,20,0,20
3,Random,Lookahead-2,0,20,0,20
4,Random,Lookahead-3,0,20,0,20
...,...,...,...,...,...,...
131,PPO_852X.pt,Lookahead-9,10,9,1,20
132,PPO_852X.pt,Lookahead-10,8,12,0,20
133,PPO_852X.pt,Lookahead-11,13,11,0,24
134,PPO_852X.pt,Lookahead-12,0,23,1,24


In [11]:
# --- Final combined report ---
print("Agent under test:", AGENT_UNDER_TEST.name)
print("\n1) Global score:", round(global_score_from_results(results, base=1.4), 5))

print("\n2) Elo-like league rating:")
print(df_rating.to_string(index=False))


Agent under test: PPO_852X.pt

1) Global score: 0.45851

2) Elo-like league rating:
       agent    elo_like
Lookahead-13 1472.467573
Lookahead-12 1439.076531
Lookahead-11 1404.553875
Lookahead-10 1358.967715
 Lookahead-9 1329.260544
 Lookahead-7 1248.502209
 Lookahead-8 1223.545751
 PPO_852X.pt 1195.016817
 Lookahead-6 1194.629500
 Lookahead-5 1186.480792
 Lookahead-4 1072.948882
 Lookahead-3  994.836560
 Lookahead-2  987.335680
 Lookahead-1  715.384476
      Center  226.332780
    Leftmost   62.841114
      Random -112.180799


In [12]:
# --- Export one-row evaluation summary to Excel ("Eval summary.xlsx") ---
#
# Keep only the useful stuff:
#   - headline: elo_like, global_score
#   - win-rates vs benchmark opponents: Random/Center/Leftmost + LA-*
#
# Regret / blunder columns were intentionally removed.

from pathlib import Path
import pandas as pd
import numpy as np
import re

EXCEL_PATH = Path("Eval summary.xlsx")
SHEET_NAME = "Summary"

def _safe_get(d, key, default=np.nan):
    try:
        return d.get(key, default)
    except Exception:
        return default

def _label_to_col(label: str) -> str:
    """Map benchmark labels to friendly column names."""
    s = str(label)
    if s == "Random":
        return "Random"
    if s == "Center":
        return "Center"
    if s == "Leftmost":
        return "Leftmost"
    m = re.match(r"Lookahead-(\d+)$", s)
    if m:
        return f"LA-{int(m.group(1))}"
    return s.replace("wr_", "").replace("_", "-")

def _flatten_current_benchmark(results: dict) -> dict:
    """Flat columns with friendly names: Random, Center, Leftmost, LA-1 ..."""
    flat = {}
    for label, st in results.items():
        col = _label_to_col(label)
        flat[col] = float(_safe_get(st, "win_rate", np.nan))
    return flat

def _get_elo_for_agent(df_rating: pd.DataFrame, agent_name: str) -> float:
    if df_rating is None or len(df_rating) == 0:
        return float("nan")
    if "agent" not in df_rating.columns:
        return float("nan")
    row = df_rating[df_rating["agent"] == agent_name]
    if len(row) == 0:
        return float("nan")
    if "elo_like" in df_rating.columns:
        return float(row.iloc[0]["elo_like"])
    if "elo" in df_rating.columns:
        return float(row.iloc[0]["elo"])
    return float("nan")

def _la_key(col: str) -> int:
    m = re.match(r"LA-(\d+)$", str(col))
    return int(m.group(1)) if m else 10**9

# ------------- Build one summary row -------------
agent_name = getattr(AGENT_UNDER_TEST, "name", "UnknownAgent")

row = {
    "elo_like": _get_elo_for_agent(df_rating if "df_rating" in globals() else None, agent_name),
    "global_score": float(G) if "G" in globals() else float("nan"),
}

if "results" in globals():
    row.update(_flatten_current_benchmark(results))

df_new = pd.DataFrame([row], index=pd.Index([agent_name], name="agent"))

# ------------- Read existing -------------
if EXCEL_PATH.exists():
    try:
        df_old = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME, index_col=0)
        df_old.index.name = "agent"
    except Exception:
        df_old = pd.DataFrame()
else:
    df_old = pd.DataFrame()

# Ensure consistent columns (union)
all_cols = sorted(set(df_old.columns).union(df_new.columns))
df_old = df_old.reindex(columns=all_cols)
df_new = df_new.reindex(columns=all_cols)

# Append time-series (duplicate index values allowed)
df_out = pd.concat([df_old, df_new], axis=0)

# ------------- Column order: elo, global, Random/Center/Leftmost, LA-1..LA-13, then rest -------------
cols = list(df_out.columns)

head = [c for c in ["elo_like", "global_score"] if c in cols]
basics = [c for c in ["Random", "Center", "Leftmost"] if c in cols]
la_cols = sorted([c for c in cols if re.match(r"LA-\d+$", str(c))], key=_la_key)
rest = [c for c in cols if c not in (head + basics + la_cols)]

ordered = head + basics + la_cols + rest
df_out = df_out.reindex(columns=ordered)

# ------------- Write Excel -------------
with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl", mode="w") as w:
    df_out.to_excel(w, sheet_name=SHEET_NAME, index=True)

print(f"Wrote {EXCEL_PATH} | rows now: {len(df_out)} | appended agent index: {agent_name}")


Wrote Eval summary.xlsx | rows now: 210 | appended agent index: PPO_852X.pt


In [13]:
# --- Excel formatting: blue header, wider columns, filters, freeze panes (row + first column),
#     and number formatting (2 decimals) ---

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
import datetime as _dt

EXCEL_PATH = "Eval summary.xlsx"
SHEET_NAME = "Summary"

wb = load_workbook(EXCEL_PATH)
ws = wb[SHEET_NAME]

# Header styling
header_fill = PatternFill(fill_type="solid", fgColor="1F4E79")  # dark-ish blue
header_font = Font(bold=True, color="FFFFFF")
header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)

# Apply style to header row
for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = header_align

ws.row_dimensions[1].height = 22

# Freeze header row AND first column (agent index column)
ws.freeze_panes = "B2"

# Enable filters (auto-filter over the used range)
ws.auto_filter.ref = ws.dimensions

# Number formatting
# - Default numeric: 2 decimals
NUM_FMT_2DP = "0.000"

# Apply numeric format to all numeric cells (excluding header)
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=2, max_col=ws.max_column):
    for cell in row:
        v = cell.value
        # Skip blanks and non-numbers; treat bool separately (bool is subclass of int in Python)
        if v is None or isinstance(v, bool):
            continue
        if isinstance(v, (int, float, np.integer, np.floating)):
            cell.number_format = NUM_FMT_2DP

# Column width auto-fit (approx; Excel has no true autofit via openpyxl)
min_w = 10
max_w = 45
pad = 2

for col_idx in range(1, ws.max_column + 1):
    col_letter = get_column_letter(col_idx)
    max_len = 0
    for row_idx in range(1, ws.max_row + 1):
        v = ws.cell(row=row_idx, column=col_idx).value
        if v is None:
            continue
        s = str(v)
        if len(s) > max_len:
            max_len = len(s)
    width = max(min_w, min(max_w, max_len + pad))
    ws.column_dimensions[col_letter].width = width

ws.sheet_view.showGridLines = True

wb.save(EXCEL_PATH)
print(f"Formatted: {EXCEL_PATH} ({SHEET_NAME})")


Formatted: Eval summary.xlsx (Summary)


In [14]:
end_time = time.time()
total_elapsed = (end_time - start_time) / 60
print(f"Evaluation completed in {total_elapsed:.1f} minutes")

Evaluation completed in 19.2 minutes
